## 2018

In [2]:
import os 
import time
import glob
import csv
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

# === ฟังก์ชันรอการดาวน์โหลดไฟล์ ===
def wait_for_download(path, timeout=60):
    start = time.time()
    while time.time() - start < timeout:
        files = glob.glob(os.path.join(path, "*.xls*"))
        if files:
            latest = max(files, key=os.path.getctime)
            if not latest.endswith(".crdownload"):
                return latest
        time.sleep(1)
    return None

# === ตั้งค่าโฟลเดอร์ดาวน์โหลด (โฟลเดอร์ 2018) ===
download_dir = os.path.abspath("2018")
os.makedirs(download_dir, exist_ok=True)

options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

# URL หลัก
base_url = "https://www.eppo.go.th/epposite/index.php/th/petroleum/price/oil-price?orders[publishUp]=publishUp&issearch=1"

# === เงื่อนไขเลือกปั๊ม ===
target_stations = ["ปตท", "บางจาก", "เชลล์"]

# === ไฟล์ CSV output ===
output_file = os.path.join(download_dir, "oil_price_E10_2018.csv")
with open(output_file, mode="w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["date", "station", "fuel_type", "effective_date", "price"])
    writer.writeheader()

    # === ช่วงเวลา 01/01/2018 → 31/12/2018 ===
    start_date = datetime.strptime("01/01/2018", "%d/%m/%Y")
    end_date   = datetime.strptime("31/12/2018", "%d/%m/%Y")

    for d in range((end_date - start_date).days + 1):
        date = (start_date + timedelta(days=d)).strftime("%d/%m/%Y")
        print(f"📌 Processing date: {date}")

        driver.get(base_url)

        # === ตรวจสอบ iframe และหา TbxToDate ===
        iframes = driver.find_elements(By.TAG_NAME, "iframe")
        date_box = None
        for i, iframe in enumerate(iframes):
            driver.switch_to.frame(iframe)
            try:
                date_box = WebDriverWait(driver, 5).until(
                    EC.presence_of_element_located((By.ID, "TbxToDate"))
                )
                print(f"✅ เจอ TbxToDate ใน iframe {i}")
                break
            except:
                driver.switch_to.default_content()

        if not date_box:
            print("❌ ไม่เจอช่องใส่วันที่")
            continue

        # ✅ กรอกวันที่
        date_box.clear()
        date_box.send_keys(date)

        # ✅ กดปุ่ม Generate
        btn = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.ID, "BtnGenerate"))
        )
        driver.execute_script("arguments[0].click();", btn)

        driver.switch_to.default_content()

        # === รอไฟล์ดาวน์โหลด ===
        file_path = wait_for_download(download_dir, timeout=60)
        if not file_path:
            print(f"❌ ไม่มีไฟล์ดาวน์โหลดของ {date}")
            continue
        print(f"✅ เจอไฟล์ดาวน์โหลด: {file_path}")

        # === 1) แปลง Excel → CSV ===
        if file_path.endswith(".xls"):
            df = pd.read_excel(file_path, engine="xlrd", header=None)
        else:
            df = pd.read_excel(file_path, engine="openpyxl", header=None)

        temp_csv = file_path.replace(".xls", ".csv").replace(".xlsx", ".csv")
        df.to_csv(temp_csv, index=False, encoding="utf-8-sig")
        print(f"📂 แปลง Excel → CSV: {temp_csv}")

        # === 2) อ่าน CSV ใหม่ และ drop NaN ===
        csv_df = pd.read_csv(temp_csv, header=None)
        csv_df = csv_df.dropna(how="all").reset_index(drop=True)

        # === หา header row (ต้องมี PTT/ปตท) ===
        header_row = csv_df[csv_df.astype(str).apply(
            lambda r: any(x in str(r.values) for x in ["PTT", "ปตท"]), axis=1
        )]
        if header_row.empty:
            print("⚠️ ไม่พบ header row")
            continue
        stations = [str(x).strip() for x in header_row.iloc[0, 1:]]

        # === หา fuel row ของ Gasohol 95-E10 ===
        fuel_row = csv_df[csv_df.astype(str).apply(
            lambda r: "95" in str(r[0]) and "E10" in str(r[0]), axis=1
        )]
        if fuel_row.empty:
            print(f"⚠️ ไม่พบ Gasohol 95 E10 ใน {temp_csv}")
            continue
        fuel_type = fuel_row.iloc[0, 0]
        prices = fuel_row.iloc[0, 1:].tolist()

        # === หา effective date row ===
        eff_row = csv_df[csv_df.astype(str).apply(
            lambda r: "Effective Date" in str(r[0]), axis=1
        )]
        eff_dates = list(eff_row.iloc[0, 1:]) if not eff_row.empty else [""] * len(stations)

        # === กรองเฉพาะปั๊มที่ต้องการ ===
        for st, pr, eff in zip(stations, prices, eff_dates):
            if pd.isna(pr) or pr == "-" or st == "nan":
                continue
            if st not in target_stations:
                continue

            writer.writerow({
                "date": datetime.strptime(date, "%d/%m/%Y").strftime("%Y-%m-%d"),
                "station": st,
                "fuel_type": fuel_type,
                "effective_date": eff,
                "price": pr
            })

driver.quit()
print(f"🎉 Done! All results saved to {output_file}")


📌 Processing date: 01/01/2018
✅ เจอ TbxToDate ใน iframe 1
✅ เจอไฟล์ดาวน์โหลด: c:\zzz\oil\scrape\2018\EPPO_RetailOilPrice_on_20180101.xls
📂 แปลง Excel → CSV: c:\zzz\oil\scrape\2018\EPPO_RetailOilPrice_on_20180101.csv
📌 Processing date: 02/01/2018
✅ เจอ TbxToDate ใน iframe 1
✅ เจอไฟล์ดาวน์โหลด: c:\zzz\oil\scrape\2018\EPPO_RetailOilPrice_on_20180101.xls
📂 แปลง Excel → CSV: c:\zzz\oil\scrape\2018\EPPO_RetailOilPrice_on_20180101.csv
📌 Processing date: 03/01/2018
✅ เจอ TbxToDate ใน iframe 1
✅ เจอไฟล์ดาวน์โหลด: c:\zzz\oil\scrape\2018\EPPO_RetailOilPrice_on_20180102.xls
📂 แปลง Excel → CSV: c:\zzz\oil\scrape\2018\EPPO_RetailOilPrice_on_20180102.csv
📌 Processing date: 04/01/2018
✅ เจอ TbxToDate ใน iframe 1
✅ เจอไฟล์ดาวน์โหลด: c:\zzz\oil\scrape\2018\EPPO_RetailOilPrice_on_20180103.xls
📂 แปลง Excel → CSV: c:\zzz\oil\scrape\2018\EPPO_RetailOilPrice_on_20180103.csv
📌 Processing date: 05/01/2018
✅ เจอ TbxToDate ใน iframe 1
✅ เจอไฟล์ดาวน์โหลด: c:\zzz\oil\scrape\2018\EPPO_RetailOilPrice_on_20180103.xls

In [4]:
import pandas as pd

# === อ่าน CSV เดิม ===
file_path = r"C:\zzz\oil\scrape\2018\oil_price_E10_2018.csv"
df = pd.read_csv(file_path)

# === ลบ NaN ออก ===
df = df.dropna()

# === จัดรูปแบบวันที่หลัก ===
df["date"] = pd.to_datetime(df["date"]).dt.date

# === effective_date ไม่ต้องแปลง datetime ตรง ๆ ===
# แค่เก็บเป็น string แล้ว strip เว้นวรรค
df["effective_date"] = df["effective_date"].astype(str).str.strip()

# === Pivot Table ===
df_pivot = df.pivot_table(
    index=["date", "effective_date"],
    columns="station",
    values="price",
    aggfunc="first"
).reset_index()

# === ตั้งชื่อคอลัมน์ใหม่ ===
df_pivot = df_pivot.rename(
    columns={
        "บางจาก": "price_Bangchak",
        "ปตท": "price_PTT",
        "เชลล์": "price_Shell"
    }
)

# === Save CSV ===
save_path = r"C:\zzz\oil\scrape\oil_price_E10_2018.csv"
df_pivot.to_csv(save_path, index=False, encoding="utf-8-sig")

print(f"บันทึกไฟล์เรียบร้อยแล้วที่: {save_path}")


บันทึกไฟล์เรียบร้อยแล้วที่: C:\zzz\oil\scrape\oil_price_E10_2018.csv


In [1]:
import pandas as pd
import os

# === Path ===
input_path = r"C:\zzz\oil\scrape\use\data\oil_price_E10_2018.csv"
output_dir = r"C:\zzz\oil\scrape\use\data_filled"
output_path = os.path.join(output_dir, "oil_price_E10_2018.csv")

# === อ่านไฟล์ ===
df = pd.read_csv(input_path)

# === เติมค่าช่องว่างด้วย forward fill + backward fill ===
df_filled = df.ffill().bfill()

# === สร้างโฟลเดอร์ปลายทางถ้ายังไม่มี ===
os.makedirs(output_dir, exist_ok=True)

# === บันทึกไฟล์ใหม่ ===
df_filled.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"บันทึกไฟล์เรียบร้อยแล้วที่: {output_path}")


บันทึกไฟล์เรียบร้อยแล้วที่: C:\zzz\oil\scrape\use\data_filled\oil_price_E10_2018.csv


In [8]:
import pandas as pd
import numpy as np
import os

# === ฟังก์ชันทำความสะอาดไฟล์ราคาน้ำมัน ===
def clean_oil_price(input_path, output_path):
    # อ่านไฟล์
    df = pd.read_csv(input_path)

    # แปลง date
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # แปลง effective_date → เติม year จาก date
    df["effective_date"] = pd.to_datetime(
        df["effective_date"].astype(str) + " " + df["date"].dt.year.astype(str),
        errors="coerce"
    )

    # แทนค่า 0 ด้วย NaN
    for col in ["price_Bangchak", "price_PTT", "price_Shell"]:
        if col in df.columns:
            df[col] = df[col].replace(0, np.nan)

    # กรองเฉพาะแถวที่ effective_date <= date
    df = df[df["effective_date"] <= df["date"]]

    # เลือก effective_date ล่าสุดของแต่ละวัน
    df_clean = (
        df.sort_values(["date", "effective_date"])
          .groupby("date")
          .tail(1)
    )

    # เติมค่าที่หายไปด้วย forward fill
    df_clean = df_clean.sort_values("date").ffill()

    # จัด column order (เพื่อความสวยงาม)
    cols = ["date", "price_Bangchak", "price_PTT", "price_Shell", "effective_date"]
    df_clean = df_clean[[c for c in cols if c in df_clean.columns]]

    # บันทึกไฟล์ใหม่
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_clean.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"✅ Cleaned and saved: {output_path}")
    return df_clean


# === Run loop ทุกปี 2018–2022 ===
input_dir = r"C:\zzz\oil\scrape\use\data"
output_dir = r"C:\zzz\oil\scrape\use\data_filled"

years = [2018, 2019, 2020, 2021, 2022]

for year in years:
    input_path = os.path.join(input_dir, f"oil_price_E10_{year}.csv")
    output_path = os.path.join(output_dir, f"oil_price_E10_{year}.csv")

    if os.path.exists(input_path):
        clean_oil_price(input_path, output_path)
    else:
        print(f"⚠️ File not found: {input_path}")


✅ Cleaned and saved: C:\zzz\oil\scrape\use\data_filled\oil_price_E10_2018.csv
✅ Cleaned and saved: C:\zzz\oil\scrape\use\data_filled\oil_price_E10_2019.csv
✅ Cleaned and saved: C:\zzz\oil\scrape\use\data_filled\oil_price_E10_2020.csv
✅ Cleaned and saved: C:\zzz\oil\scrape\use\data_filled\oil_price_E10_2021.csv
✅ Cleaned and saved: C:\zzz\oil\scrape\use\data_filled\oil_price_E10_2022.csv


In [ ]:
import pandas as pd
import os

def build_full_year(input_path, output_path, last_price=None):
    # อ่านไฟล์จาก data_filled (ที่ clean แล้ว)
    df = pd.read_csv(input_path, parse_dates=["date", "effective_date"])

    # ปี
    year = df["date"].dt.year.iloc[0]

    # สร้างช่วงวันทั้งปี
    full_range = pd.date_range(start=f"{year}-01-01", end=f"{year}-12-31", freq="D")
    df_full = pd.DataFrame({"date": full_range})

    # รวมกับราคาที่ clean แล้ว
    df_merged = pd.merge(df_full, df, on="date", how="left")

    # เติมค่าด้วย forward fill
    df_merged = df_merged.ffill()

    # เติมราคาปีที่แล้วสำหรับวันแรกของปี (ถ้าไม่มีประกาศใหม่)
    if last_price is not None:
        for col in ["price_Bangchak", "price_PTT", "price_Shell"]:
            if col in df_merged.columns and pd.isna(df_merged[col].iloc[0]):
                df_merged.at[0, col] = last_price[col]

    # บันทึก
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_merged.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"✅ Built full year: {output_path}")

    return df_merged


✅ Cleaned: C:\zzz\oil\scrape\use\data_last\oil_price_E10_2018.csv
✅ Cleaned: C:\zzz\oil\scrape\use\data_last\oil_price_E10_2019.csv
✅ Cleaned: C:\zzz\oil\scrape\use\data_last\oil_price_E10_2020.csv
✅ Cleaned: C:\zzz\oil\scrape\use\data_last\oil_price_E10_2021.csv
✅ Cleaned: C:\zzz\oil\scrape\use\data_last\oil_price_E10_2022.csv


In [14]:
import pandas as pd
import numpy as np
import os

def clean_and_fill_full_year(input_path, output_path, last_price=None):
    # === Step 1: อ่านไฟล์ raw ===
    df = pd.read_csv(input_path)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["effective_date"] = pd.to_datetime(
        df["effective_date"].astype(str) + " " + df["date"].dt.year.astype(str),
        errors="coerce"
    )

    # === Step 2: แทนค่า 0 ด้วย NaN ===
    for col in ["price_Bangchak", "price_PTT", "price_Shell"]:
        if col in df.columns:
            df[col] = df[col].replace(0, np.nan)

    # === Step 3: เลือก record ล่าสุดของแต่ละวัน (effective_date ≤ date) ===
    df = df[df["effective_date"] <= df["date"]]
    df_clean = (
        df.sort_values(["date", "effective_date"])
          .groupby("date")
          .tail(1)
          .sort_values("date")
          .ffill()
    )

    # === Step 4: สร้างช่วงวันที่เต็ม (365/366 วัน) ===
    year = df_clean["date"].dt.year.iloc[0]
    full_range = pd.date_range(start=f"{year}-01-01", end=f"{year}-12-31", freq="D")
    df_full = pd.DataFrame({"date": full_range})

    # === Step 5: merge กับข้อมูลจริงที่ clean แล้ว ===
    df_final = pd.merge(df_full, df_clean, on="date", how="left")

    # === Step 6: เติมด้วย forward fill (ราคาต่อเนื่อง) ===
    df_final = df_final.ffill()

    # === Step 7: เติมราคาสุดท้ายของปีก่อนให้วันแรก ถ้าไม่มี ===
    if last_price is not None:
        for col in ["price_Bangchak", "price_PTT", "price_Shell"]:
            if col in df_final.columns and pd.isna(df_final[col].iloc[0]):
                df_final.at[0, col] = last_price[col]

    # === Step 8: บันทึก ===
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_final.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"✅ Cleaned & filled full year: {output_path}")

    return df_final



input_dir = r"C:\zzz\oil\scrape\use\data"
output_dir = r"C:\zzz\oil\scrape\use\data_last"

years = [2018, 2019, 2020, 2021, 2022]
last_price = None

for year in years:
    input_path = os.path.join(input_dir, f"oil_price_E10_{year}.csv")
    output_path = os.path.join(output_dir, f"oil_price_E10_{year}.csv")

    if os.path.exists(input_path):
        df_year = clean_and_fill_full_year(input_path, output_path, last_price=last_price)

        # เก็บราคาสุดท้ายของปีนี้ → ใช้เป็น base สำหรับปีถัดไป
        last_price = {
            col: df_year[col].iloc[-1]
            for col in ["price_Bangchak", "price_PTT", "price_Shell"]
            if col in df_year.columns
        }



✅ Cleaned & filled full year: C:\zzz\oil\scrape\use\data_last\oil_price_E10_2018.csv
✅ Cleaned & filled full year: C:\zzz\oil\scrape\use\data_last\oil_price_E10_2019.csv
✅ Cleaned & filled full year: C:\zzz\oil\scrape\use\data_last\oil_price_E10_2020.csv
✅ Cleaned & filled full year: C:\zzz\oil\scrape\use\data_last\oil_price_E10_2021.csv
✅ Cleaned & filled full year: C:\zzz\oil\scrape\use\data_last\oil_price_E10_2022.csv


In [ ]:
import pandas as pd
import glob
import os

# path input (ไฟล์ดิบ) และ output (ไฟล์ clean)
input_path = r"C:\zzz\oil\scrape\use\data"
output_path = r"C:\zzz\oil\scrape\use\data_enhance"

# สร้างโฟลเดอร์ output ถ้ายังไม่มี
os.makedirs(output_path, exist_ok=True)

# ดึงทุกไฟล์ที่เป็น oil_price_E10_xxxx.csv
files = glob.glob(os.path.join(input_path, "oil_price_E10_*.csv"))

for file in files:
    print(f"📂 กำลังประมวลผลไฟล์: {file}")
    
    df = pd.read_csv(file)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.sort_values("date").reset_index(drop=True)

    # เติมราคา forward fill + backward fill
    df[["price_Bangchak", "price_PTT", "price_Shell"]] = (
        df[["price_Bangchak", "price_PTT", "price_Shell"]]
        .ffill()
        .bfill()
    )

    # เติม effective_date เช่นกัน
    df["effective_date"] = df["effective_date"].ffill().bfill()

    # สร้างชื่อไฟล์ใหม่
    filename = os.path.basename(file).replace(".csv", "_filled.csv")
    output_file = os.path.join(output_path, filename)

    # บันทึกไฟล์
    df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"✅ บันทึกไฟล์เรียบร้อย: {output_file}")

print("🎉 เสร็จสิ้นทุกไฟล์")


FileNotFoundError: [Errno 2] No such file or directory: 'oil_price_E10_2021_filled.csv'

In [17]:
import pandas as pd
import glob
import os

# === path input/output ===
input_path = r"C:\zzz\oil\scrape\use\data_enhance"   # ไฟล์ที่ fill แล้ว
output_path = r"C:\zzz\oil\scrape\use\data_enhance\clean_2018_2022"
os.makedirs(output_path, exist_ok=True)

# === loop ทุกปีที่ต้องการ ===
for year in range(2018, 2023):   # 2018 → 2022
    file = os.path.join(input_path, f"oil_price_E10_{year}_filled.csv")
    if not os.path.exists(file):
        print(f"⚠️ ไม่มีไฟล์ {file}, ข้าม...")
        continue
    
    print(f"📂 กำลังทำไฟล์: {file}")
    df = pd.read_csv(file)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    
    # ✅ ลบ duplicate date
    df = df.drop_duplicates(subset=["date"], keep="last")
    
    # ✅ สร้างช่วงวันครบ 1 ปี (365/366 วัน)
    start_date = f"{year}-01-01"
    end_date   = f"{year}-12-31"
    full_range = pd.date_range(start=start_date, end=end_date, freq="D")
    
    # ✅ align กับ full_range
    df = df.set_index("date").reindex(full_range).reset_index()
    df = df.rename(columns={"index": "date"})
    
    # ✅ เติมราคาด้วย forward/backward fill
    df[["price_Bangchak", "price_PTT", "price_Shell"]] = (
        df[["price_Bangchak", "price_PTT", "price_Shell"]].ffill().bfill()
    )
    df["effective_date"] = df["effective_date"].ffill().bfill()
    
    # ✅ export ไฟล์ใหม่
    output_file = os.path.join(output_path, f"oil_price_E10_{year}_clean.csv")
    df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"✅ บันทึกไฟล์: {output_file}")

print("🎉 เสร็จสิ้น: ทำความสะอาดข้อมูลปี 2018–2022 เรียบร้อย")


📂 กำลังทำไฟล์: C:\zzz\oil\scrape\use\data_enhance\oil_price_E10_2018_filled.csv
✅ บันทึกไฟล์: C:\zzz\oil\scrape\use\data_enhance\clean_2018_2022\oil_price_E10_2018_clean.csv
📂 กำลังทำไฟล์: C:\zzz\oil\scrape\use\data_enhance\oil_price_E10_2019_filled.csv
✅ บันทึกไฟล์: C:\zzz\oil\scrape\use\data_enhance\clean_2018_2022\oil_price_E10_2019_clean.csv
📂 กำลังทำไฟล์: C:\zzz\oil\scrape\use\data_enhance\oil_price_E10_2020_filled.csv
✅ บันทึกไฟล์: C:\zzz\oil\scrape\use\data_enhance\clean_2018_2022\oil_price_E10_2020_clean.csv
📂 กำลังทำไฟล์: C:\zzz\oil\scrape\use\data_enhance\oil_price_E10_2021_filled.csv
✅ บันทึกไฟล์: C:\zzz\oil\scrape\use\data_enhance\clean_2018_2022\oil_price_E10_2021_clean.csv
📂 กำลังทำไฟล์: C:\zzz\oil\scrape\use\data_enhance\oil_price_E10_2022_filled.csv
✅ บันทึกไฟล์: C:\zzz\oil\scrape\use\data_enhance\clean_2018_2022\oil_price_E10_2022_clean.csv
🎉 เสร็จสิ้น: ทำความสะอาดข้อมูลปี 2018–2022 เรียบร้อย


## 2019

In [1]:
import os 
import time
import glob
import csv
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

# === ฟังก์ชันรอการดาวน์โหลดไฟล์ ===
def wait_for_download(path, timeout=60):
    start = time.time()
    while time.time() - start < timeout:
        files = glob.glob(os.path.join(path, "*.xls*"))
        if files:
            latest = max(files, key=os.path.getctime)
            if not latest.endswith(".crdownload"):
                return latest
        time.sleep(1)
    return None

# === ตั้งค่าโฟลเดอร์ดาวน์โหลด (โฟลเดอร์ 2019) ===
download_dir = os.path.abspath("2019")
os.makedirs(download_dir, exist_ok=True)

options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

# URL หลัก
base_url = "https://www.eppo.go.th/epposite/index.php/th/petroleum/price/oil-price?orders[publishUp]=publishUp&issearch=1"

# === เงื่อนไขเลือกปั๊ม ===
target_stations = ["ปตท", "บางจาก", "เชลล์"]

# === ไฟล์ CSV output ===
output_file = os.path.join(download_dir, "oil_price_E10_2019.csv")
with open(output_file, mode="w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["date", "station", "fuel_type", "effective_date", "price"])
    writer.writeheader()

    # === ช่วงเวลา 01/01/2019 → 31/12/2019 ===
    start_date = datetime.strptime("01/01/2019", "%d/%m/%Y")
    end_date   = datetime.strptime("31/12/2019", "%d/%m/%Y")

    for d in range((end_date - start_date).days + 1):
        date = (start_date + timedelta(days=d)).strftime("%d/%m/%Y")
        print(f"📌 Processing date: {date}")

        driver.get(base_url)

        # === ตรวจสอบ iframe และหา TbxToDate ===
        iframes = driver.find_elements(By.TAG_NAME, "iframe")
        date_box = None
        for i, iframe in enumerate(iframes):
            driver.switch_to.frame(iframe)
            try:
                date_box = WebDriverWait(driver, 5).until(
                    EC.presence_of_element_located((By.ID, "TbxToDate"))
                )
                print(f"✅ เจอ TbxToDate ใน iframe {i}")
                break
            except:
                driver.switch_to.default_content()

        if not date_box:
            print("❌ ไม่เจอช่องใส่วันที่")
            continue

        # ✅ กรอกวันที่
        date_box.clear()
        date_box.send_keys(date)

        # ✅ กดปุ่ม Generate
        btn = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.ID, "BtnGenerate"))
        )
        driver.execute_script("arguments[0].click();", btn)

        driver.switch_to.default_content()

        # === รอไฟล์ดาวน์โหลด ===
        file_path = wait_for_download(download_dir, timeout=60)
        if not file_path:
            print(f"❌ ไม่มีไฟล์ดาวน์โหลดของ {date}")
            continue
        print(f"✅ เจอไฟล์ดาวน์โหลด: {file_path}")

        # === 1) แปลง Excel → CSV ===
        if file_path.endswith(".xls"):
            df = pd.read_excel(file_path, engine="xlrd", header=None)
        else:
            df = pd.read_excel(file_path, engine="openpyxl", header=None)

        temp_csv = file_path.replace(".xls", ".csv").replace(".xlsx", ".csv")
        df.to_csv(temp_csv, index=False, encoding="utf-8-sig")
        print(f"📂 แปลง Excel → CSV: {temp_csv}")

        # === 2) อ่าน CSV ใหม่ และ drop NaN ===
        csv_df = pd.read_csv(temp_csv, header=None)
        csv_df = csv_df.dropna(how="all").reset_index(drop=True)

        # === หา header row (ต้องมี PTT/ปตท) ===
        header_row = csv_df[csv_df.astype(str).apply(
            lambda r: any(x in str(r.values) for x in ["PTT", "ปตท"]), axis=1
        )]
        if header_row.empty:
            print("⚠️ ไม่พบ header row")
            continue
        stations = [str(x).strip() for x in header_row.iloc[0, 1:]]

        # === หา fuel row ของ Gasohol 95-E10 ===
        fuel_row = csv_df[csv_df.astype(str).apply(
            lambda r: "95" in str(r[0]) and "E10" in str(r[0]), axis=1
        )]
        if fuel_row.empty:
            print(f"⚠️ ไม่พบ Gasohol 95 E10 ใน {temp_csv}")
            continue
        fuel_type = fuel_row.iloc[0, 0]
        prices = fuel_row.iloc[0, 1:].tolist()

        # === หา effective date row ===
        eff_row = csv_df[csv_df.astype(str).apply(
            lambda r: "Effective Date" in str(r[0]), axis=1
        )]
        eff_dates = list(eff_row.iloc[0, 1:]) if not eff_row.empty else [""] * len(stations)

        # === กรองเฉพาะปั๊มที่ต้องการ ===
        for st, pr, eff in zip(stations, prices, eff_dates):
            if pd.isna(pr) or pr == "-" or st == "nan":
                continue
            if st not in target_stations:
                continue

            writer.writerow({
                "date": datetime.strptime(date, "%d/%m/%Y").strftime("%Y-%m-%d"),
                "station": st,
                "fuel_type": fuel_type,
                "effective_date": eff,
                "price": pr
            })

driver.quit()
print(f"🎉 Done! All results saved to {output_file}")


📌 Processing date: 01/01/2019
✅ เจอ TbxToDate ใน iframe 1
✅ เจอไฟล์ดาวน์โหลด: c:\zzz\oil\scrape\2019\EPPO_RetailOilPrice_on_20190101.xls
📂 แปลง Excel → CSV: c:\zzz\oil\scrape\2019\EPPO_RetailOilPrice_on_20190101.csv
📌 Processing date: 02/01/2019
✅ เจอ TbxToDate ใน iframe 1
✅ เจอไฟล์ดาวน์โหลด: c:\zzz\oil\scrape\2019\EPPO_RetailOilPrice_on_20190101.xls
📂 แปลง Excel → CSV: c:\zzz\oil\scrape\2019\EPPO_RetailOilPrice_on_20190101.csv
📌 Processing date: 03/01/2019
✅ เจอ TbxToDate ใน iframe 1
✅ เจอไฟล์ดาวน์โหลด: c:\zzz\oil\scrape\2019\EPPO_RetailOilPrice_on_20190102.xls
📂 แปลง Excel → CSV: c:\zzz\oil\scrape\2019\EPPO_RetailOilPrice_on_20190102.csv
📌 Processing date: 04/01/2019
✅ เจอ TbxToDate ใน iframe 1
✅ เจอไฟล์ดาวน์โหลด: c:\zzz\oil\scrape\2019\EPPO_RetailOilPrice_on_20190103.xls
📂 แปลง Excel → CSV: c:\zzz\oil\scrape\2019\EPPO_RetailOilPrice_on_20190103.csv
📌 Processing date: 05/01/2019
✅ เจอ TbxToDate ใน iframe 1
✅ เจอไฟล์ดาวน์โหลด: c:\zzz\oil\scrape\2019\EPPO_RetailOilPrice_on_20190104.xls

In [5]:
import pandas as pd

# === อ่าน CSV เดิม ===
file_path = r"C:\zzz\oil\scrape\2019\oil_price_E10_2019.csv"
df = pd.read_csv(file_path)

# === ลบ NaN ออก ===
df = df.dropna()

# === จัดรูปแบบวันที่หลัก ===
df["date"] = pd.to_datetime(df["date"]).dt.date

# === effective_date ไม่ต้องแปลง datetime ตรง ๆ ===
# แค่เก็บเป็น string แล้ว strip เว้นวรรค
df["effective_date"] = df["effective_date"].astype(str).str.strip()

# === Pivot Table ===
df_pivot = df.pivot_table(
    index=["date", "effective_date"],
    columns="station",
    values="price",
    aggfunc="first"
).reset_index()

# === ตั้งชื่อคอลัมน์ใหม่ ===
df_pivot = df_pivot.rename(
    columns={
        "บางจาก": "price_Bangchak",
        "ปตท": "price_PTT",
        "เชลล์": "price_Shell"
    }
)

# === Save CSV ===
save_path = r"C:\zzz\oil\scrape\oil_price_E10_2019.csv"
df_pivot.to_csv(save_path, index=False, encoding="utf-8-sig")

print(f"บันทึกไฟล์เรียบร้อยแล้วที่: {save_path}")


บันทึกไฟล์เรียบร้อยแล้วที่: C:\zzz\oil\scrape\oil_price_E10_2019.csv
